[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jac99/pw-llm-lab/blob/main/labs/lab3/lab3_finetuning_lecture.ipynb)

# Lab 3 – Dostosowywanie generatywnych modeli językowych (notatnik wykładowy)

**Cele:**
- stosować in-context learning (zero-shot i few-shot) z modelem generatywnym
- dostrajać model metodami PEFT: LoRA, prompt tuning, p-tuning
- porównać jakość i koszt metod dostrajania

**Wymagania:** GPU T4 (*Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*).  
**Szacowany czas:** ~1,5–2 h, w tym ~25 min treningów na GPU  
**Ocena:** notatnik wykładowy – bez oceny

Zacznij od uruchomienia komórki *Konfiguracja środowiska* poniżej.

In [ ]:
# @title Konfiguracja środowiska (uruchom jako pierwszą)
%pip install -q "transformers==4.57.3" "peft==0.17.1" "bitsandbytes==0.48.2" "datasets==4.4.1" "accelerate==1.11.0" "colorama==0.4.6"
import os
import random

import numpy as np
import torch

# Tryb skrócony (mniej danych i kroków), używany m.in. w testach automatycznych.
FAST_MODE = os.environ.get("FAST_MODE", "0") == "1"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    assert FAST_MODE, "Brak GPU. Włącz: Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU"
print("PyTorch", torch.__version__, "| FAST_MODE:", FAST_MODE)

Notatnik omawia:

*   Wykorzystanie generatywnego modelu językowego **Phi-1.5** z biblioteki Hugging Face
*   **In-context learning** (zero-shot prompting, few-shot prompting)
*   Wybrane *metody efektywnego dostrajania PEFT* (*Parameter-Efficient Fine-Tuning*)

Model językowy **Phi-1.5** jest modelem klasy tylko-dekoder zwanej również przyczynowym modelem językowym (*causal language model*). Modele tej klasy są trenowane do predykcji prawdopodobieństwa kolejnego tokenu na podstawie sekwencji poprzedzających tokenów.
Termin **przyczynowy** oznacza, że w trakcie uczenia model dla każdego tokenu w sekwencji treningowej estymuje rozkład prawdopodobieństwa kolejnego tokenu biorąc pod uwagę tylko tokeny występujące NIE później w sekwencji.
Modele tej klasy zawierają tylko bloki dekodera Transformer z przyczynową maską atencji. Do tej klasy należą współczesne generatywne modele językowe z rodziny GPT, Llama i Mistral.

Model **Phi-1.5 oparty jest o architekturę Transformer i posiada 1,3 miliarda parametrów**. Dzięki wykorzystaniu wysokiej jakości zbioru treningowego osiąga wyniki porównywalne z modelami o dziesięciokrotnie większej liczbie parametrów w popularnych benchmarkach rozumienia tekstu w języku naturalnym. Phi-1.5 jest bazowym modelem językowym i nie był dostrajany do wykonywania poleceń ani do wykorzystania w systemach dialogowych.
Opis modelu na stronie Hugging Face: [link](https://huggingface.co/microsoft/phi-1_5).

W serwisie Hugging Face dostępne są również większe modele: **Phi-2** z 2,7 miliardami parametrów, **Phi-3** (m.in. Phi-3-mini-instruct z 3,8 miliardami parametrów i Phi-3-vision-instruct), **Phi-3.5** (m.in. Phi-3.5-mini-instruct i Phi-3.5-MoE-instruct) oraz **Phi-4** (14 miliardów parametrów) i **Phi-4-mini** (3,8 miliarda parametrów).

# 1 Przygotowanie środowiska

In [ ]:
!nvidia-smi

Pakiety wykorzystywane w notatniku:


*   datasets (Hugging Face) - umożliwia dostęp do zbiorów danych (audio, wizja komputerowa, NLP) [link](https://huggingface.co/docs/datasets/index)
*   transformers (Hugging Face) - zawiera wstępnie wytrenowane modele oraz narzędzia do efektywnego dostrajania modeli głębokich sieci neuronowych (NLP, wizja komputerowa) opartych o architekturę Transformer [link](https://huggingface.co/docs/transformers/index)
*   peft (Hugging Face) - **metody efektywnego dostrajania dużych modeli językowych** (PEFT, *Parameter-Efficient Fine-Tuning*) [link](https://huggingface.co/docs/peft/index)
*   bitsandbytes - **biblioteka do kwantyzacji** sieci neuronowych do 8 lub 4 bitów [link](https://github.com/TimDettmers/bitsandbytes)



In [ ]:
import transformers
import datasets
import peft
import bitsandbytes

In [ ]:
print(f"Wersja transformers: {transformers.__version__}")
print(f"Wersja datasets: {datasets.__version__}")
print(f"Wersja PEFT: {peft.__version__}")
print(f"Wersja bitsandbytes: {bitsandbytes.__version__}")

In [ ]:
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 2 Przygotowanie modelu językowego

## 2.1 Utworzenie instancji wstępnie wytrenowanego modelu językowego Phi-1.5

Aby utworzyć instancję wstępnie wytrenowanego modelu, wykorzystamy funkcję `AutoModelForCausalLM.from_pretrained()`.
W celu zmniejszenia wymagań na zasoby obliczeniowe wykorzystamy bibliotekę BitsAndBytes, aby skwantyzować wagi modelu do 4 bitów (format NF4, *4-bit NormalFloat*, wykorzystywany w metodzie QLoRA – patrz rozdział 4.3).

W trybie `FAST_MODE` (testy automatyczne na CPU) zamiast Phi-1.5 wczytywany jest mały, losowo zainicjalizowany model o tej samej architekturze, bez kwantyzacji – jego odpowiedzi są przypadkowe.

In [ ]:
# Na CPU (FAST_MODE, testy automatyczne) zamiast Phi-1.5 używamy małego, losowo zainicjalizowanego
# modelu o tej samej architekturze (PhiForCausalLM, te same nazwy warstw).
base_model_id = "hf-internal-testing/tiny-random-PhiForCausalLM" if FAST_MODE else "microsoft/phi-1_5"
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_base_model(quantize: bool = True):
    """Wczytuje model bazowy. Na GPU (quantize=True) w kwantyzacji 4-bitowej NF4, tak jak w metodzie QLoRA.
    Na CPU (FAST_MODE) bez kwantyzacji – bitsandbytes wymaga GPU."""
    if quantize and torch.cuda.is_available():
        q_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",             # 4-bitowy typ NormalFloat (NF4)
            bnb_4bit_compute_dtype=torch.float16,  # obliczenia w fp16 (T4 nie obsługuje bf16)
        )
        return AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config,
                                                    dtype=torch.float16, device_map={"": torch.cuda.current_device()})  # indeks GPU, nie "cuda" – wymaga tego accelerate przy treningu modelu 4-bit
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    return AutoModelForCausalLM.from_pretrained(base_model_id, dtype=dtype).to(device)


model = load_base_model()

In [ ]:
print(model)

In [ ]:
print(f"Zajętość pamięci przez model: {model.get_memory_footprint() / 1024**3:0.3f} GiB")

W wersji nieskwantyzowanej, w pełnej 32-bitowej precyzji (FP32), wagi modelu zajęłyby około 5,7 GB (ok. 5,3 GiB) pamięci GPU – dla porównania komórka powyżej wypisuje w GiB zajętość wersji skwantyzowanej.

## 2.2 Utworzenie instancji tokenizatora

Model Phi-1.5 wykorzystuje tokenizator CodeGen oparty o metodę Byte-Pair Encoding (BPE). Więcej informacji na temat tokenizacji i opis wybranych metod tokenizacji w serwisie Hugging Face: [link](https://huggingface.co/docs/transformers/tokenizer_summary).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

print(f"Rozmiar słownika: {tokenizer.vocab_size}")

Sprawdzenie działania tokenizatora - zakodowanie wejściowego tekstu i zdekodowanie otrzymanych tokenów. Możemy zauważyć, że niektóre tokeny zawierają wiodącą spację.

In [ ]:
s = "A small dog is running across the street. A black cat is sleeping."
results = tokenizer(s)
print(results)
print()

# Dekodowanie kolejnych tokenów
print("Zdekodowane tokeny:")
for token_id in results['input_ids']:
  txt = tokenizer.decode(token_id)
  print(f"{token_id:10} |{txt}|")

## 2.3 Sprawdzenie działania modelu językowego

Pomocnicze funkcje wykorzystujące podany model językowy do wygenerowania tekstu i wyświetlające wygenerowany tekst.
Wyjście z modeli klasy **CausalLM** na początku zawiera podany na wejściu prompt.

In [ ]:
from colorama import Fore
from transformers import BatchEncoding


def generate_text(model: nn.Module, model_input: BatchEncoding, max_new_tokens: int = 100,
                  return_full_text: bool = False) -> str:
  # Generowanie tekstu przez model

  model.eval()
  with torch.no_grad():
    generated_tokens = model.generate(
        input_ids = model_input['input_ids'],
        attention_mask = model_input['attention_mask'],
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,  # zakończ generowanie po wygenerowaniu tokenu EOS
        pad_token_id=tokenizer.pad_token_id,
        )[0]
    # generated_tokens zawiera tokeny wejściowe i nowo wygenerowane
    if not return_full_text:
      # Tylko nowo wygenerowane tokeny
      generated_tokens = generated_tokens[model_input['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def generate_and_print_text(model: nn.Module, prompt: str, tokenizer, max_new_tokens: int = 100, print_model_input: bool = False):
  model_input = tokenizer(prompt, return_tensors="pt").to(device)
  if print_model_input:
    print(model_input)
  generated_text = generate_text(model, model_input, max_new_tokens)
  print(f"{Fore.RESET}{prompt}", end="")
  print(f"{Fore.BLUE}{generated_text}{Fore.RESET}")

Sprawdzenie działania modelu dla przykładowych promptów.

In [ ]:
prompt = "Write an analogy between Transformer and a polar bear."
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70, print_model_input=True)

In [ ]:
prompt = "The highest building in the world is"
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70, print_model_input=True)

Model Phi-1.5 oprócz tekstu w języku naturalnym potrafi również generować kod w języku Python.

In [ ]:
prompt = """def count_primes(n):
   # Count the number of primes between 1 and n
   """
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=160, print_model_input=True)

In [ ]:
prompt = "Give me a step-by-step recipe how to make a pancake."
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=100, print_model_input=True)

Model Phi-1.5 jest modelem bazowym, wstępnie wytrenowanym na zadaniu przyczynowego modelowania języka, i nie został dostrojony do wykonywania poleceń i udzielania odpowiedzi zgodnych z preferencjami użytkownika. W związku z tym generowane odpowiedzi na pytania często nie są najlepsze.

## 2.4 Uczenie z kontekstu (*in-context learning*)

**Uczenie z kontekstu (*in-context learning*)** pozwala na wpływanie na działanie modelu poprzez podanie odpowiedniego promptu (kontekstu), bez dostrajania modelu czy modyfikowania jego wag. Dzięki temu odpowiedzi modelu będą bardziej zgodne z oczekiwaniami użytkownika.

Technika **few-shot prompting** polega na skonstruowaniu promptu zawierającego **przykłady demonstrujące oczekiwane odpowiedzi**.
Pozwala poprawić jakość odpowiedzi dużych modeli językowych bez konieczności trenowania czy dostrajania modelu [(link)](https://arxiv.org/abs/2005.14165).

Na przykład, chcielibyśmy, aby model językowy określił **sentyment** (wydźwięk emocjonalny) podanego tekstu w języku naturalnym.
Jeśli w prompcie podamy sam tekst, który chcemy ocenić, np. *'What a horrible show'*, model językowy, który nie został dostrojony do wykonywania poleceń, nie domyśli się, o co nam chodzi, i będzie generował tekst będący kontynuacją wejścia.

***k*-shot prompting** oznacza podanie w prompcie $k$ przykładów demonstrujących poprawną odpowiedź.
Szczególnym przykładem jest ***0*-shot prompting** oznaczający prompt bez przykładów demonstrujących poprawną odpowiedź - zawierający tylko polecenie mające wymusić od modelu oczekiwane zachowanie.

In [ ]:
prompt= "What a horrible show"

generate_and_print_text(model, prompt, tokenizer, max_new_tokens=40)

Możemy spróbować poprosić model, aby zrobił to, czego oczekujemy (*zero-shot prompting*).
Jeśli jednak model językowy nie był dostrajany, aby wykonywać polecenia (a model Phi-1.5 był w bardzo ograniczonym zakresie), to możemy nie uzyskać oczekiwanej odpowiedzi.

In [ ]:
prompt= "Answer in one word. What's the sentiment of the following sentence: What a horrible show."

generate_and_print_text(model, prompt, tokenizer, max_new_tokens=40)

Sprawdź, czy model odpowiedział jednym słowem. Model bazowy, który nie był dostrajany do wykonywania poleceń, często ignoruje takie instrukcje – np. powtarza fragment polecenia albo generuje dalszy ciąg tekstu.

Efekt powinien być znacznie lepszy, jeśli w prompcie zademonstrujemy oczekiwane odpowiedzi (*few-shot prompting*).

Aby oddzielić zdanie i jego sentyment w podawanych w prompcie przykładach, użyjemy separatora "`//`". Nie ma znaczenia, jakiego separatora użyjemy; ważne, aby wybrana sekwencja znaków nie występowała w normalnym tekście.

In [ ]:
demonstration= """This is awesome // Positive
This is bad // Negative
Wow that movie was bad // Negative
"""

prompt= demonstration + "What a horrible show //"
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=3)

prompt= demonstration + "The movie was not so bad //"
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=3)

Sprawdź, czy podanie w prompcie (kontekście) kilku przykładów demonstrujących oczekiwany wynik pozwoliło uzyskać odpowiedzi w oczekiwanym formacie – czy model generuje w odpowiedzi tylko słowo `Positive` albo `Negative`?

# 3 Przygotowanie zbioru danych

W notatniku zostanie wykorzystany zbiór danych **e2e** zawierający opinie o lokalach gastronomicznych w języku naturalnym i zawarte w nich informacje w ustandaryzowanym formacie. Artykuł opisujący zbiór danych: [link](https://arxiv.org/abs/1706.09254).

In [ ]:
from datasets import load_dataset

dataset_name = 'marcov/e2e_nlg_cleaned_promptsource'
# Mały zbiór walidacyjny (200 przykładów) skraca ewaluację w trakcie treningu.
# W FAST_MODE (testy na CPU) korzystamy z małych wycinków zbiorów.
n_train, n_val, n_test = (64, 16, 8) if FAST_MODE else (10000, 200, 1000)
train_dataset = load_dataset(dataset_name, split=f'train[:{n_train}]')
val_dataset = load_dataset(dataset_name, split=f'validation[:{n_val}]')
test_dataset = load_dataset(dataset_name, split=f'test[:{n_test}]')

In [ ]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

Element zbioru danych jest słownikiem. W notatniku wykorzystamy następujące pola:

*   **human_reference** - opis lokalu gastronomicznego w języku naturalnym
*   **meaning_representation** - sformalizowana reprezentacja semantyki (znaczenia) opisu. Reprezentacja jest listą złożoną z elementów postaci `atrybut[wartość atrybutu]`, gdzie `atrybut` należy do zbioru: `name`, `eatType`, `food`, `priceRange`, `customer rating`, `area`, `familyFriendly`, `near`.



In [ ]:
ndx = 1
for key in ['human_reference', 'meaning_representation']:
    print(f"{key}: {train_dataset[ndx][key]}")

## 3.1 Problem text2table

**Zadanie polega na dostrojeniu modelu Phi-1.5 do ekstrakcji informacji w ustrukturyzowanej formie z opisu lokalu gastronomicznego w języku naturalnym (zadanie text2table)**.
Na przykład, dla opisu:

*Located near the Rainbow Vegetarian Café, it's inexpensive and family friendly,  but the Chinese food at Strada has a low customer rating*

powinna zostać wygenerowana ustrukturyzowana reprezentacja (zwana *meaning representation*):

*name[Strada], food[Chinese], priceRange[less than £20], customer rating[low], familyFriendly[yes], near[Rainbow Vegetarian Café]*

Aby wymusić generowanie przez model odpowiedzi w oczekiwanym formacie, musimy skonstruować odpowiedni prompt, który podamy na wejście modelu.
Prompt testowy (`test_prompt_with_instruction`) zawiera jedynie polecenie do wykonania i opis lokalu – wykorzystamy go do sprawdzenia modelu.
Prompt treningowy (`train_prompt_with_instruction`) zawiera dodatkowo oczekiwaną odpowiedź – wykorzystamy go jako przykład demonstracyjny w *few-shot prompting*.
Przy dostrajaniu modelu na wejście podamy prompt testowy i oczekiwaną odpowiedź zakończoną tokenem końca tekstu (EOS), a funkcję straty będziemy liczyć tylko na odpowiedzi (rozdział 3.2).

In [ ]:
instruction = """Construct the underlying meaning representation of the input sentence as a list of attributes and their values.
The output must be a comma separated list of attributes and their values formatted as: attribute[value].
The attribute name must be one of the following: 'name', 'eatType', 'familyFriendly', 'priceRange', 'food', 'near', 'area', 'customer rating'."""

def train_prompt_with_instruction(data_point) -> str:
    prompt =f"""{instruction}
### Input sentence:
{data_point["human_reference"]}
### Meaning representation:
{data_point["meaning_representation"]}
---"""
    return prompt

In [ ]:
print("Element zbioru danych:")
print(train_dataset[1])
print()

print("Prompt wykorzystywany do trenowania modelu:")
print(train_prompt_with_instruction(train_dataset[1]))

In [ ]:
def test_prompt_with_instruction(data_point) -> str:
    prompt =f"""{instruction}
### Input sentence:
{data_point["human_reference"]}
### Meaning representation:
"""
    return prompt

In [ ]:
print("Element zbioru danych:")
print(test_dataset[1])
print()

print("Prompt wykorzystywany do testowania modelu:")
print(test_prompt_with_instruction(test_dataset[1]))

## 3.2 Tokenizacja zbiorów danych

Funkcja pomocnicza tokenizująca pojedynczy tekst (bez wyrównywania długości). Wykorzystamy ją m.in. do sprawdzania długości promptów.

In [ ]:
def tokenize(prompt: str) -> dict[str, Any]:
    # Tokenizator zwraca słownik z polami input_ids i attention_mask.
    return tokenizer(prompt)

Funkcja pomocnicza wykorzystywana do tokenizacji przykładów treningowych. Aby podczas trenowania modelu można było utworzyć wsady złożone z kilku elementów, konieczne jest wyrównanie długości tekstów po tokenizacji – każdy przykład jest uzupełniany (*padding*) albo obcinany do długości `max_length`.

Funkcja `tokenize_completion_only` tokenizuje osobno prompt (polecenie i pytanie) oraz oczekiwaną odpowiedź:
* do odpowiedzi dodaje token końca tekstu (EOS), dzięki czemu model uczy się kończyć generowanie po odpowiedzi,
* etykiety (`labels`) tokenów promptu i wypełnienia ustawia na -100, więc funkcja straty jest liczona tylko na odpowiedzi.

Tak przygotowane przykłady nie wymagają dodatkowego przetwarzania – w klasie `Trainer` wystarczy `data_collator=transformers.default_data_collator`. (Kolektor `DataCollatorForLanguageModeling` nadpisałby etykiety i zamaskował również token EOS, bo jest on jednocześnie tokenem wypełnienia.)

In [ ]:
def tokenize_completion_only(prompt: str, answer: str, max_length: int) -> dict[str, list[int]]:
    """Tokenizuje prompt i odpowiedź do stałej długości `max_length`.

    - Na końcu odpowiedzi dodawany jest token EOS, dzięki czemu model uczy się kończyć generowanie.
    - Etykiety (`labels`) promptu i wypełnienia (padding) mają wartość -100, więc funkcja straty
      jest liczona tylko na tokenach odpowiedzi (przesunięcie o jeden token wykonuje sam model).
    """
    prompt_ids = tokenizer(prompt)["input_ids"]
    answer_ids = tokenizer(answer + tokenizer.eos_token)["input_ids"]
    input_ids = (prompt_ids + answer_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]
    n_pad = max_length - len(input_ids)
    return {
        "input_ids": input_ids + [tokenizer.pad_token_id] * n_pad,
        "attention_mask": [1] * len(input_ids) + [0] * n_pad,
        "labels": labels + [-100] * n_pad,
    }

Długość `max_length` ustalimy na podstawie rozkładu długości przykładów treningowych (99. percentyl), tak aby obcinana była tylko niewielka część przykładów. Następnie wykorzystamy metodę `map`, aby zastosować funkcję `generate_and_tokenize_train_prompt_with_padding` dla każdego elementu zbioru danych.

In [ ]:
def n_tokens(prompt: str, answer: str) -> int:
    # Liczba tokenów przykładu treningowego: prompt + odpowiedź + EOS
    return len(tokenize(prompt)["input_ids"]) + len(tokenize(answer + tokenizer.eos_token)["input_ids"])


lengths = np.array([n_tokens(test_prompt_with_instruction(e), e["meaning_representation"]) for e in train_dataset])
max_length = int(np.ceil(np.percentile(lengths, 99)))
print(f"max_length (99. percentyl): {max_length}, obcinanych przykładów: {np.mean(lengths > max_length):.1%}")


def generate_and_tokenize_train_prompt_with_padding(data_point):
    return tokenize_completion_only(test_prompt_with_instruction(data_point), data_point["meaning_representation"], max_length)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_train_prompt_with_padding, remove_columns=train_dataset.column_names)
tokenized_val_dataset = val_dataset.map(generate_and_tokenize_train_prompt_with_padding, remove_columns=val_dataset.column_names)

In [ ]:
# Sprawdzenie (na najkrótszym przykładzie, który na pewno nie został obcięty): etykiety promptu
# i wypełnienia są równe -100, a odpowiedź kończy się tokenem EOS.
i = int(np.argmin(lengths))
item = tokenized_train_dataset[i]
n_prompt = len(tokenize(test_prompt_with_instruction(train_dataset[i]))["input_ids"])
answer_labels = [t for t in item["labels"] if t != -100]
assert all(t == -100 for t in item["labels"][:n_prompt])
assert answer_labels[-1] == tokenizer.eos_token_id
print(item["input_ids"])
print("Tokeny, na których liczona jest funkcja straty:", repr(tokenizer.decode(answer_labels)))

# 4 Wykorzystanie pretrenowanego modelu językowego Phi-1.5


## 4.1 Zero-shot learning

Sprawdźmy, czy model jest w stanie zwrócić poprawną odpowiedź w oczekiwanym formacie po podaniu na wejściu prompta zawierającego polecenie. Nie będziemy wykorzystywali żadnych przykładów uczących (*zero-shot learning*).

In [ ]:
prompt = test_prompt_with_instruction(test_dataset[1])
print("Prompt:")
print(prompt)
print()
print("Wygenerowana odpowiedź:")
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70)

Oceń uzyskaną odpowiedź: czy zawiera oczekiwane nazwy atrybutów, czy ich wartości są poprawne i czy formatowanie jest zgodne z oczekiwanym?

## 4.2 Few-shot prompting


Sprawdźmy, czy technika *few-shot prompting* pozwoli poprawić uzyskane wyniki.
Skonstruujemy prompt zawierający kilka przykładów z oczekiwanymi odpowiedziami.

In [ ]:
import random

random.seed(123)


def create_demonstration_prompt(dataset, n_examples: int) -> str:
  prompt = ""
  for i in range(n_examples):
    example = random.choice(dataset)
    prompt += train_prompt_with_instruction(example)
    prompt += "\n"

  return prompt

Demonstracja z jednym przykładem (*one-shot prompting*).

In [ ]:
s = create_demonstration_prompt(train_dataset, 1)
s += test_prompt_with_instruction(test_dataset[1])

generate_and_print_text(model, s, tokenizer, max_new_tokens=50)

Sprawdź, czy format wygenerowanego tekstu jest zgodny z oczekiwaniami i czy wartości atrybutów są poprawne – modele często podają w takiej sytuacji niepoprawne wartości atrybutów lub konfabulują.
Spróbujemy dodać do promptu więcej przykładów ze zbioru treningowego (*few-shot prompting*).

In [ ]:
s = create_demonstration_prompt(train_dataset, 4)
s += test_prompt_with_instruction(test_dataset[1])

generate_and_print_text(model, s, tokenizer, max_new_tokens=60)

Porównaj odpowiedź z odpowiedzią uzyskaną dla jednego przykładu demonstracyjnego. W celu ilościowej oceny uzyskanych wyników należałoby napisać kod do ewaluacji odpowiedzi na zbiorze testowym (w laboratorium robimy to dla zadania ekstrakcji danych w formacie JSON – funkcja `evaluate_json`). Tutaj pozostawimy to jako zadanie dla chętnych.

## 4.3 Dostrajanie modelu metodą LoRA

Zamiast szukać promptu, który pozwoli uzyskać jak najlepsze wyniki, możemy dostroić model, aby generował oczekiwane wyniki w sposób nadzorowany na zbiorze treningowym.
W tej części notatnika sprawdzimy **metodę efektywnego dostrajania modelu zwaną LoRA** (*Low-Rank Adaptation of Large Language Models*), a właściwie jej skwantyzowaną wersję **QLoRA**: wagi modelu bazowego są zamrożone i skwantyzowane do 4 bitów (NF4), a trenowane są tylko adaptery LoRA w wyższej precyzji. Z elementów QLoRA wykorzystamy kwantyzację NF4 i stronicowany optymalizator (*paged optimizer*); pomijamy podwójną kwantyzację (*double quantization*).
Opis metody LoRA w serwisie Hugging Face: [link](https://huggingface.co/docs/peft/task_guides/lora_based_methods).
Artykuł opisujący metodę QLoRA: [link](https://huggingface.co/papers/2305.14314).

In [ ]:
from peft import prepare_model_for_kbit_training

# Wykorzystamy model bazowy wczytany w rozdziale 2 (na GPU: 4-bit NF4).
# prepare_model_for_kbit_training zamraża wagi, rzutuje nieskwantyzowane parametry (normalizacje, embeddingi, lm_head) do fp32
# i włącza gradient checkpointing (mniejsze zużycie pamięci kosztem czasu).
if getattr(model, "is_loaded_in_4bit", False):
    model = prepare_model_for_kbit_training(model)
print(model)

Utworzenie instancji modelu przygotowanej do dostrojenia metodą LoRA. Dokumentacja w serwisie Hugging Face: [link](https://huggingface.co/docs/peft/en/package_reference/lora).

**Uwaga:** `get_peft_model` modyfikuje przekazany model w miejscu – dodaje do jego warstw adaptery LoRA (kopia modelu, np. `copy.deepcopy(model)`, podwoiłaby zajętość pamięci). Odpowiedzi modelu bazowego uzyskamy, tymczasowo wyłączając adapter:
```python
with peft_model.disable_adapter():
    generate_and_print_text(peft_model, prompt, tokenizer)
```

Model skwantyzowany do 4 bitów przed dodaniem adapterów przygotowujemy funkcją `prepare_model_for_kbit_training` (zamrożenie wag, rzutowanie pozostałych, nieskwantyzowanych parametrów – normalizacji, embeddingów, `lm_head` – do fp32, włączenie *gradient checkpointing*). W FAST_MODE (CPU) model nie jest skwantyzowany i ten krok jest pomijany.

Nazwy warstw, do których dodajemy adaptery LoRA (parametr `target_modules`), zależą od implementacji modelu. Wypiszmy nazwy warstw liniowych (`nn.Linear`) w modelu Phi. W modelu skwantyzowanym są to warstwy `bitsandbytes` (`Linear4bit`), które dziedziczą po `nn.Linear`.

In [ ]:
linear_layer_names = sorted({name.split(".")[-1] for name, module in model.named_modules()
                             if isinstance(module, nn.Linear)})
print(linear_layer_names)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Adaptery w warstwach atencji (q_proj, k_proj, v_proj, dense) i MLP (fc1, fc2); pomijamy lm_head
config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
                    bias="none", lora_dropout=0.05, task_type=TaskType.CAUSAL_LM)

# Utworzenie modelu do efektywnego dostrajania (adaptery LoRA są dodawane do modelu `model` w miejscu)
peft_model = get_peft_model(model, config)

Sprawdzenie liczby dostrajanych parametrów w modelu trenowanym z wykorzystaniem metody LoRA.

In [ ]:
peft_model.print_trainable_parameters()

In [ ]:
print(peft_model)

### 4.3.1 Trenowanie modelu

Do dostrojenia modelu `peft_model` wykorzystamy klasę Trainer implementującą pętlę treningową modeli w PyTorch. Klasa Trainer wspiera m.in. efektywne rozproszone trenowanie na wielu GPU/TPU i trenowanie w mieszanej precyzji (*mixed precision*). Więcej informacji: [link](https://huggingface.co/docs/transformers/main_classes/trainer)

Ponieważ trenowany model został wcześniej skwantyzowany, do optymalizacji parametrów sieci wykorzystamy 8-bitową wersję optymalizatora Paged AdamW (`paged_adamw_8bit`, wymaga GPU – w FAST_MODE na CPU używamy `adamw_torch`).

Trening trwa 300 kroków (efektywny rozmiar wsadu 12, czyli ok. 3600 przykładów) z typową dla LoRA stałą uczenia 2e-4. Co 100 kroków liczona jest wartość funkcji straty na małym zbiorze walidacyjnym. Checkpointy nie są zapisywane (`save_strategy="no"`) – po treningu zapiszemy sam adapter. Na GPU T4 trening powinien zająć kilkanaście minut (szacunek).

In [ ]:
import time

import transformers

lora_output_dir = "./phi-qlora-e2e"  # osobny katalog dla każdej metody

report_to = "none"
#report_to = "wandb"

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=lora_output_dir,
        per_device_train_batch_size=2 if FAST_MODE else 6,
        gradient_accumulation_steps=1 if FAST_MODE else 2,  # efektywny rozmiar wsadu: 6 × 2 = 12
        max_steps=4 if FAST_MODE else 300,
        learning_rate=2e-4,                                 # typowa stała uczenia dla LoRA
        warmup_steps=0 if FAST_MODE else 10,
        optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
        fp16=torch.cuda.is_available(),                     # mieszana precyzja na GPU
        logging_steps=1 if FAST_MODE else 25,               # funkcja straty na zbiorze treningowym co 25 kroków
        eval_strategy="steps",
        eval_steps=2 if FAST_MODE else 100,                 # funkcja straty na zbiorze walidacyjnym co 100 kroków
        per_device_eval_batch_size=8,
        save_strategy="no",                                 # bez checkpointów – adapter zapiszemy po treningu
        report_to=report_to,
        seed=SEED,
    ),
    data_collator=transformers.default_data_collator,
)

peft_model.config.use_cache = False  # Wyłącza ostrzeżenia (włączamy ponownie po treningu, przed inferencją)
start = time.time()
trainer.train()
print(f"Czas treningu: {time.time() - start:.0f} s")
peft_model.config.use_cache = True

Zapisanie adaptera LoRA metodą `save_pretrained` – zapisywane są tylko wagi adaptera (kilka–kilkanaście MB), a nie cały model. Aby sprawdzić, że zapis jest kompletny, zwolnimy pamięć, wczytamy ponownie model bazowy i dołączymy do niego zapisany adapter (`PeftModel.from_pretrained`).

Pliki zapisane w Colabie znikają po zakończeniu sesji – aby zachować adapter, skopiuj katalog np. na Dysk Google.

In [ ]:
import gc

from peft import PeftModel

peft_model.save_pretrained(lora_output_dir)

# Zwolnienie pamięci
del peft_model, model, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Ponowne wczytanie: model bazowy + zapisany adapter
model = load_base_model()
peft_model = PeftModel.from_pretrained(model, lora_output_dir)

### 4.3.2 Sprawdzenie dostrojonego modelu

In [ ]:
prompt = test_prompt_with_instruction(test_dataset[1])
print("Model dostrojony:")
generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=64)

# Odpowiedź modelu bazowego – adapter LoRA tymczasowo wyłączony
print("\nModel bazowy:")
with peft_model.disable_adapter():
    generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=64)

In [ ]:
prompt = test_prompt_with_instruction(test_dataset[5])
generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=128)

## 4.4 Soft prompts

Dostrajanie dużego modelu językowego wymaga znacznych zasobów obliczeniowych. Alternatywnym podejściem jest zamrożenie wag modelu i przygotowanie odpowiedniego promptu tak, aby model wygenerował oczekiwaną odpowiedź. Stosowane są dwa podejścia:

1.   **Hard prompts** - wykorzystanie promptu w języku naturalnym.
2.   **Soft prompts** - wykorzystanie wirtualnego promptu dołączonego do sekwencji wektorowych reprezentacji wejściowych tokenów. Wirtualny prompt jest tworzony w przestrzeni wektorowych reprezentacji tokenów (embeddingów) - NIE MA reprezentacji w języku naturalnym. Jest optymalizowany w procesie uczenia tak, by zminimalizować wartość funkcji straty na zbiorze treningowym.

Do metod klasy *soft prompts* należą: *prompt tuning*, *prefix tuning* i *p-tuning*. Więcej informacji: [link](https://huggingface.co/docs/peft/conceptual_guides/prompting)


### 4.4.1 Metoda dostrajania promptu



W tej części notatnika sprawdzimy metodę dostrajania promptu (*prompt tuning*). W tej metodzie do sekwencji wektorowych reprezentacji wejściowych tokenów dołączany jest wirtualny prompt.
**Wagi modelu** w procesie uczenia **pozostają zamrożone, a dostrajane są tylko wartości wirtualnego promptu**. Uczenie odbywa się w standardowy sposób metodą stochastycznego spadku wzdłuż gradientu. Artykuł opisujący metodę prompt tuning: [link](https://huggingface.co/papers/2104.08691)

**Uwaga:**
Wirtualny prompt jest tworzony w przestrzeni wektorowych reprezentacji słów (jest wektorem o współrzędnych rzeczywistych) a nie w przestrzeni identyfikatorów tokenów (liczb naturalnych). Nie może on zostać zinterpretowany w języku naturalnym.

Wygeneruj ponownie zbiór treningowy i walidacyjny. Nie dodawaj polecenia "*Construct the underlying meaning representation...*" do wejściowych promptów.
Zamiast polecenia w języku naturalnym wykorzystamy wirtualny prompt wyznaczony w procesie uczenia.
Wirtualny prompt może zostać zainicjalizowany losowo lub podanym tekstem w języku naturalnym.



In [ ]:
def test_prompt(data_point) -> str:
  prompt =f"""### Input sentence:
{data_point["human_reference"]}
### Meaning representation:
"""
  return prompt


def train_prompt(data_point) -> str:
  # Prompt testowy i oczekiwana odpowiedź (przy trenowaniu dodawany jest jeszcze token EOS)
  return test_prompt(data_point) + data_point["meaning_representation"]

In [ ]:
print("Prompt wykorzystywany w treningu modelu:")
print(train_prompt(train_dataset[1]))
print("")

print("Prompt wykorzystywany podczas inferencji:")
print(test_prompt(train_dataset[1]))
print("")

Ponownie przygotuj zbiór treningowy i walidacyjny po tokenizacji, korzystając z nowego promptu.


In [ ]:
lengths = np.array([n_tokens(test_prompt(e), e["meaning_representation"]) for e in train_dataset])
p = 99
print(f"Liczba tokenów w przykładzie – min: {np.min(lengths)}   średnia: {np.mean(lengths):0.1f}   maks.: {np.max(lengths)}   {p}. percentyl: {np.percentile(lengths, p):0.1f}")

In [ ]:
max_length = int(np.ceil(np.percentile(lengths, 99)))
print(f"max_length: {max_length}, obcinanych przykładów: {np.mean(lengths > max_length):.1%}")


def generate_and_tokenize_train_prompt_with_padding(data_point):
  return tokenize_completion_only(test_prompt(data_point), data_point["meaning_representation"], max_length)

tokenized_train_dataset = train_dataset.map(generate_and_tokenize_train_prompt_with_padding, remove_columns=train_dataset.column_names)
tokenized_val_dataset = val_dataset.map(generate_and_tokenize_train_prompt_with_padding, remove_columns=val_dataset.column_names)

Utworzenie instancji modelu przygotowanej do dostrojenia metodą *prompt tuning*. Dokumentacja w serwisie Hugging Face: [link](https://huggingface.co/docs/peft/main/en/task_guides/prompt_based_methods?configurations=prompt+tuning).

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit, TaskType

# Inicjalizacja wirtualnego promptu
initial_prompt = "Hocus-pocus: 'name', 'eatType', 'familyFriendly', 'priceRange', 'food', 'near', 'area', 'customer rating'"
t = tokenize(initial_prompt)
initial_prompt_len = len(t['input_ids'])
print(f"Długość początkowego promptu (w tokenach): {initial_prompt_len}")

peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.TEXT, prompt_tuning_init_text=initial_prompt,
                                 tokenizer_name_or_path=base_model_id, num_virtual_tokens=initial_prompt_len)

#peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.RANDOM, num_virtual_tokens=initial_prompt_len)


In [ ]:
# Zwolnienie pamięci po części LoRA (model 4-bitowy z adapterem nie będzie już potrzebny)
del peft_model, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Prompt tuning: model bazowy w fp16, bez kwantyzacji. get_peft_model nie zmienia warstw modelu
# (dodaje tylko wirtualny prompt), więc kopia modelu nie jest potrzebna.
print(base_model_id)
model = load_base_model(quantize=False)
peft_model = get_peft_model(model, peft_config)
print(peft_model)

In [ ]:
peft_model.print_trainable_parameters()

Funkcja pomocnicza dekodująca wirtualny prompt poprzez znalezienie tokenów w słowniku o wektorowych reprezentacjach najbardziej zbliżonych, w sensie odległości euklidesowej, do wartości wirtualnego promptu.
Funkcja wyświetla zdekodowany prompt oraz odległość euklidesową pomiędzy każdym elementem wirtualnego promptu a najbliższym mu tokenem w słowniku.

In [ ]:
def decode_soft_prompt(peft_model: nn.Module):
  token_embeddings = peft_model.base_model.model.embed_tokens.weight.detach()
  prompt_embedding = peft_model.prompt_encoder['default'].embedding.weight.detach()
  print(f"Rozmiar macierzy embeddingów słownika: {token_embeddings.shape}")
  print(f"Rozmiar wirtualnego promptu: {prompt_embedding.shape}")

  token_ids = []
  for i in range(len(prompt_embedding)):
    dist = torch.norm(token_embeddings - prompt_embedding[i], dim=-1)
    out = torch.min(dist, dim=0)
    min_ndx = out[1].item()
    print(f"{i}: odległość do tokenu {min_ndx} |{tokenizer.decode([min_ndx])}|: {out[0]:0.4f}")
    token_ids.append(min_ndx)

  untokenized_text = tokenizer.decode(token_ids)
  print(f"Zdekodowany prompt: {untokenized_text}")

Zdekodowanie początkowego wirtualnego promptu. Ponieważ wirtualny prompt został zainicjalizowany tekstem w języku naturalnym, odległości między każdym elementem promptu a najbliższym mu tokenem ze słownika są zerowe.

In [ ]:
decode_soft_prompt(peft_model)

Dostrojenie modelu `peft_model` z wykorzystaniem klasy `Trainer`.

Wykorzystamy optymalizator Adafactor (`adafactor`).
Wymaga on dwukrotnie mniej pamięci GPU w porównaniu do standardowo wykorzystywanego optymalizatora AdamW (AdamW: 8 bajtów na parametr; Adafactor: 4 bajty na parametr).
Artykuł opisujący sposoby optymalizacji wykorzystania pamięci i skrócenia czasu trenowania dużych modeli na pojedynczym GPU: [link](https://huggingface.co/docs/transformers/v4.37.2/en/perf_train_gpu_one).

In [ ]:
prompt_tuning_output_dir = "./phi-prompt-tuning-e2e"

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=prompt_tuning_output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1 if FAST_MODE else 4,  # efektywny rozmiar wsadu: 2 × 4 = 8
        max_steps=4 if FAST_MODE else 300,
        learning_rate=3e-4,
        warmup_steps=0 if FAST_MODE else 10,
        optim="adafactor",
        fp16=torch.cuda.is_available(),
        logging_steps=1 if FAST_MODE else 25,               # funkcja straty na zbiorze treningowym co 25 kroków
        eval_strategy="steps",
        eval_steps=2 if FAST_MODE else 100,                 # funkcja straty na zbiorze walidacyjnym co 100 kroków
        per_device_eval_batch_size=8,
        save_strategy="no",                                 # bez checkpointów – adapter zapiszemy po treningu
        report_to=report_to,
        seed=SEED,
    ),
    data_collator=transformers.default_data_collator,
)

peft_model.config.use_cache = False  # Wyłącza ostrzeżenia (włączamy ponownie po treningu, przed inferencją)
start = time.time()
trainer.train()
print(f"Czas treningu: {time.time() - start:.0f} s")
peft_model.config.use_cache = True

peft_model.save_pretrained(prompt_tuning_output_dir)  # zapis samego wirtualnego promptu (adaptera)

Odpowiedź modelu bazowego (wirtualny prompt tymczasowo wyłączony: `peft_model.disable_adapter()`).

In [ ]:
prompt = test_prompt(test_dataset[3])
with peft_model.disable_adapter():
    generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=60)

Odpowiedź z modelu `peft_model` z wykorzystaniem dostrojonego promptu.

In [ ]:
prompt = test_prompt(test_dataset[3])
generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=60)

Wykorzystaj napisaną wcześniej funkcję `decode_soft_prompt()` do zdekodowania zoptymalizowanego wirtualnego promptu zapisanego w modelu `peft_model`.

In [ ]:
decode_soft_prompt(peft_model)

Porównaj zdekodowany prompt po treningu z tekstem użytym do inicjalizacji wirtualnego promptu. Sprawdź, czy zmieniły się najbliższe tokeny ze słownika i jak zmieniły się odległości między elementami wirtualnego promptu a wektorowymi reprezentacjami najbliższych tokenów. Nawet jeśli najbliższe tokeny pozostały bez zmian, niewielka zmiana wartości wirtualnego promptu może wystarczyć, aby wpłynąć na odpowiedzi generowane przez model – porównaj je z odpowiedziami modelu bazowego powyżej.

### 4.4.2 Metoda p-tuning

W tej części notatnika sprawdzimy inną metodę z rodziny *soft prompts*, zwaną **p-tuning**.

Metoda p-tuning jest podobna do metody dostrajania promptu (*prompt tuning*). W procesie uczenia optymalizowany jest wirtualny prompt, a wagi modelu językowego pozostają zamrożone. Główne różnice w porównaniu do metody dostrajania promptu:

*   Wartości wirtualnego promptu przetwarzane są przez dodatkową sieć (domyślnie perceptron wielowarstwowy) i dopiero tak uzyskane wartości zostają połączone z wektorowymi reprezentacjami oryginalnego promptu.
*   Przetworzone przez dodatkową sieć wartości wirtualnego promptu mogą zostać wstawione w dowolne miejsce w oryginalnym prompcie.
*   Optymalizacji w procesie uczenia podlegają wartości wirtualnego promptu i wagi dodatkowej sieci.

Artykuł opisujący metodę p-tuning: [link](https://arxiv.org/abs/2103.10385).

In [ ]:
# Zwolnienie pamięci
del model
del peft_model
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Utworzenie instancji modelu przygotowanej do dostrojenia metodą p-tuning. Dokumentacja w serwisie Hugging Face: [link](https://huggingface.co/docs/peft/main/en/package_reference/p_tuning#peft.PromptEncoderConfig)

In [ ]:
from peft import PromptEncoderConfig, get_peft_model

model = load_base_model(quantize=False)

peft_config = PromptEncoderConfig(task_type="CAUSAL_LM", num_virtual_tokens=32, encoder_hidden_size=128)
peft_model = get_peft_model(model, peft_config)
peft_model.to(device)
peft_model.print_trainable_parameters()

In [ ]:
print(peft_model)

Dostrojenie modelu `peft_model` z wykorzystaniem klasy Trainer.

In [ ]:
p_tuning_output_dir = "./phi-p-tuning-e2e"

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=p_tuning_output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1 if FAST_MODE else 4,  # efektywny rozmiar wsadu: 2 × 4 = 8
        max_steps=4 if FAST_MODE else 300,
        learning_rate=3e-4,
        warmup_steps=0 if FAST_MODE else 10,
        optim="adafactor",
        fp16=torch.cuda.is_available(),
        logging_steps=1 if FAST_MODE else 25,               # funkcja straty na zbiorze treningowym co 25 kroków
        eval_strategy="steps",
        eval_steps=2 if FAST_MODE else 100,                 # funkcja straty na zbiorze walidacyjnym co 100 kroków
        per_device_eval_batch_size=8,
        save_strategy="no",                                 # bez checkpointów – adapter zapiszemy po treningu
        report_to=report_to,
        seed=SEED,
    ),
    data_collator=transformers.default_data_collator,
)

peft_model.config.use_cache = False  # Wyłącza ostrzeżenia (włączamy ponownie po treningu, przed inferencją)
start = time.time()
trainer.train()
print(f"Czas treningu: {time.time() - start:.0f} s")
peft_model.config.use_cache = True

peft_model.save_pretrained(p_tuning_output_dir)  # zapis samego wirtualnego promptu (adaptera)

Odpowiedź modelu bazowego (wirtualny prompt tymczasowo wyłączony: `peft_model.disable_adapter()`).

In [ ]:
prompt = test_prompt(test_dataset[3])
with peft_model.disable_adapter():
    generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=40)

Odpowiedź z modelu `peft_model` dostosowanego metodą p-tuning.

In [ ]:
prompt = test_prompt(test_dataset[3])
generate_and_print_text(peft_model, prompt, tokenizer, max_new_tokens=40)

# Zakończenie

Dla metod LoRA (dostrajanie modelu), prompt tuning i p-tuning (dostrajanie promptu) porównaj:
*   liczbę dostrajanych (trenowanych) parametrów
*   wartość funkcji straty na zbiorze walidacyjnym po zakończeniu uczenia (we wszystkich metodach liczona tylko na tokenach odpowiedzi, więc wartości są porównywalne)
*   czas treningu